#CVIU Experiment 3:Full D1 to D2

This notebook trains all methods on the complete D1 dataset and evaluates them on D2.

The evaluated methods are:

- **Naive Prediction**
- **ED Regression**
- **M1:Siamese**
- **M2:Siamese+Lab**
- **M3:Siamese+ΔE**
- **M4:Siamese+ΔE+Lab**

The four neural networks are implemented separately. All preprocessing limits are calculated from D1. D2 is evaluated only after neural-network training is complete and is not used for training, learning-rate scheduling, early stopping, or checkpoint selection. The final notebook output is the result table only.


##1.Setup

In [28]:
import gc
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from scipy.optimize import curve_fit
from IPython.display import display, Markdown

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
)

print("PyTorch:", torch.__version__)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

PyTorch: 2.11.0+cu128
Device: cuda


##2.Parameters

This cell contains the configurable file paths, preprocessing, architecture, training, optimizer, direct-Lab weighting, scheduler, regression, and output parameters.

In [ ]:
#File paths
D1_PATH = (
    ""
) # Please fill the path

D2_PATH = (
    ""
) # Please fill the path

OUTPUT_DIR = Path(
    ""
) # Please fill the path
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

#Random seed
SEED = 42

#Preprocessing
#D1 supplies all normalization limits.
NORMALIZATION_REFERENCE = "D1"

#Architecture
ENCODER_LAYERS = [
    32,
    64,
    64,
    64,
    64,
]

REGRESSOR_LAYERS = [
    32,
    32,
    16,
    8,
]

#Training
EPOCHS = 300
BATCH_SIZE = 8
LEARNING_RATE = 5e-4
INPUT_NOISE_STD = 0.01
WEIGHT_DECAY = 0.0
PRINT_EVERY = 25

#Direct Lab-feature weights
M2_LAB_WEIGHT = 0.01
M4_LAB_WEIGHT = 0.01

#Optional training-loss scheduler
#D2 is never used by the scheduler.
USE_SCHEDULER = False
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 50
SCHEDULER_THRESHOLD = 1e-3
MIN_LEARNING_RATE = 1e-6

#Regression and numerical stability
REGRESSION_MAX_FUNCTION_EVALUATIONS = 20000
EPSILON = 1e-7

#Output
SAVE_MODEL_WEIGHTS = True
SAVE_PREDICTIONS = True

METHOD_ORDER = [
    "Naive Prediction",
    "ED Regression",
    "Siamese",
    "Siamese+Lab",
    "Siamese+ΔE",
    "Siamese+ΔE+Lab",
]

print("Output directory:", OUTPUT_DIR)


Output directory: /content/drive/MyDrive/CVIU_Experiment3_Full_D1_to_D2


##3.Data preparation

In [30]:
def read_csv_auto(path):
    return pd.read_csv(
        path,
        sep=None,
        engine="python",
        header=0,
    )


def normalize_response_values(response_array):
    values = {
        value
        for value in pd.unique(
            response_array.ravel()
        )
        if not pd.isna(value)
    }

    if values.issubset({0, 1, 2}):
        return response_array

    if values.issubset({1, 2, 3}):
        return response_array - 1

    raise ValueError(
        "Observer responses must use either "
        "{0,1,2} or {1,2,3}. "
        f"Detected values: {sorted(values)}"
    )


def add_probability_columns(
    dataframe,
    response_columns,
):
    dataframe = dataframe.copy()

    probability_sets = [
        (
            "possibility0",
            "possibility1",
            "possibility2",
        ),
        (
            "probability0",
            "probability1",
            "probability2",
        ),
        (
            "p0",
            "p1",
            "p2",
        ),
        (
            "true1",
            "true2",
            "true3",
        ),
    ]

    for columns in probability_sets:
        if all(
            column in dataframe.columns
            for column in columns
        ):
            probabilities = dataframe.loc[
                :,
                columns,
            ].to_numpy(dtype=np.float32)

            probabilities = probabilities / np.clip(
                probabilities.sum(
                    axis=1,
                    keepdims=True,
                ),
                EPSILON,
                None,
            )

            dataframe["possibility0"] = probabilities[:, 0]
            dataframe["possibility1"] = probabilities[:, 1]
            dataframe["possibility2"] = probabilities[:, 2]

            return dataframe

    if not response_columns:
        raise ValueError(
            "No observer-response columns "
            "or probability columns were found."
        )

    responses = dataframe.loc[
        :,
        response_columns,
    ].to_numpy(dtype=np.float32)

    responses = normalize_response_values(
        responses
    )

    denominator = len(response_columns)

    for category in [0, 1, 2]:
        dataframe[
            f"possibility{category}"
        ] = (
            responses == category
        ).sum(axis=1) / denominator

    return dataframe


def detect_feature_columns(
    dataframe,
    response_columns,
):
    explicit_sets = [
        [
            "L1",
            "a1",
            "b1",
            "L2",
            "a2",
            "b2",
        ],
        [
            "L_1",
            "a_1",
            "b_1",
            "L_2",
            "a_2",
            "b_2",
        ],
        [
            "L1*",
            "a1*",
            "b1*",
            "L2*",
            "a2*",
            "b2*",
        ],
    ]

    for columns in explicit_sets:
        if all(
            column in dataframe.columns
            for column in columns
        ):
            return columns

    excluded_columns = {
        "C1",
        "C2",
        "Average",
        "possibility0",
        "possibility1",
        "possibility2",
        *response_columns,
    }

    numeric_candidates = [
        column
        for column in dataframe.columns
        if column not in excluded_columns
        and pd.api.types.is_numeric_dtype(
            dataframe[column]
        )
    ]

    if len(numeric_candidates) < 6:
        raise ValueError(
            "Could not identify six Lab columns. "
            f"Candidates: {numeric_candidates}"
        )

    return numeric_candidates[:6]


def prepare_xy(
    dataframe,
    response_columns,
):
    prepared = add_probability_columns(
        dataframe,
        response_columns,
    )

    feature_columns = detect_feature_columns(
        prepared,
        response_columns,
    )

    X = prepared.loc[
        :,
        feature_columns,
    ].to_numpy(dtype=np.float32)

    Y = prepared.loc[
        :,
        [
            "possibility0",
            "possibility1",
            "possibility2",
        ],
    ].to_numpy(dtype=np.float32)

    Y = Y / np.clip(
        Y.sum(
            axis=1,
            keepdims=True,
        ),
        EPSILON,
        None,
    )

    return (
        X,
        Y,
        prepared,
        feature_columns,
    )


d1_dataframe = read_csv_auto(
    D1_PATH
)

d2_dataframe = read_csv_auto(
    D2_PATH
)

d1_response_columns = [
    str(index)
    for index in range(1, 21)
    if str(index) in d1_dataframe.columns
]

d2_response_columns = [
    str(index)
    for index in range(1, 7)
    if str(index) in d2_dataframe.columns
]

(
    X_d1_raw,
    Y_d1,
    d1_prepared,
    D1_FEATURE_COLUMNS,
) = prepare_xy(
    d1_dataframe,
    d1_response_columns,
)

(
    X_d2_raw,
    Y_d2,
    d2_prepared,
    D2_FEATURE_COLUMNS,
) = prepare_xy(
    d2_dataframe,
    d2_response_columns,
)

assert len(X_d1_raw) == 800, (
    f"Expected 800 D1 samples, got {len(X_d1_raw)}."
)

assert len(X_d2_raw) == 102, (
    f"Expected 102 D2 samples, got {len(X_d2_raw)}."
)

FEATURE_MIN = X_d1_raw.min(
    axis=0
)

FEATURE_MAX = X_d1_raw.max(
    axis=0
)


def normalize_with_d1(X):
    return (
        X - FEATURE_MIN
    ) / (
        FEATURE_MAX
        - FEATURE_MIN
        + 1e-8
    )


X_d1 = normalize_with_d1(
    X_d1_raw
).astype(np.float32)

X_d2 = normalize_with_d1(
    X_d2_raw
).astype(np.float32)

d1_delta_e = np.linalg.norm(
    X_d1[:, :3]
    - X_d1[:, 3:],
    axis=1,
)

DELTA_E_MIN = float(
    d1_delta_e.min()
)

DELTA_E_MAX = float(
    d1_delta_e.max()
)

print("D1 features:", D1_FEATURE_COLUMNS)
print("D2 features:", D2_FEATURE_COLUMNS)
print("D1 samples:", len(X_d1))
print("D2 samples:", len(X_d2))
print("D1 target mean:", Y_d1.mean(axis=0))
print("D2 target mean:", Y_d2.mean(axis=0))

D1 features: ['L1', 'a1', 'b1', 'L2', 'a2', 'b2']
D2 features: ['L1', 'a1', 'b1', 'L2', 'a2', 'b2']
D1 samples: 800
D2 samples: 102
D1 target mean: [0.0365625 0.1408125 0.822625 ]
D2 target mean: [0.03431373 0.05718955 0.9084967 ]


##4.Shared components

In [31]:
class ColorPairDataset(Dataset):

    def __init__(
        self,
        X,
        Y,
        noise_std=0.0,
    ):
        self.X = torch.tensor(
            X,
            dtype=torch.float32,
        )

        self.Y = torch.tensor(
            Y,
            dtype=torch.float32,
        )

        self.noise_std = float(
            noise_std
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        x = self.X[index].clone()

        if self.noise_std > 0:
            x = (
                x
                + torch.randn_like(x)
                * self.noise_std
            )

        return x, self.Y[index]


class SoftCrossEntropyLoss(nn.Module):

    def forward(
        self,
        logits,
        targets,
    ):
        log_probabilities = torch.log_softmax(
            logits,
            dim=1,
        )

        return -(
            targets
            * log_probabilities
        ).sum(dim=1).mean()


criterion = SoftCrossEntropyLoss()


class SharedEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        modules = []
        input_size = 3

        for index, output_size in enumerate(
            ENCODER_LAYERS
        ):
            modules.append(
                nn.Linear(
                    input_size,
                    output_size,
                )
            )

            if index < len(ENCODER_LAYERS) - 1:
                modules.append(
                    nn.ReLU()
                )

            input_size = output_size

        self.network = nn.Sequential(
            *modules
        )

    def forward(self, x):
        return self.network(x)


def build_regressor(input_size):
    modules = []
    current_size = input_size

    for hidden_size in REGRESSOR_LAYERS:
        modules.append(
            nn.Linear(
                current_size,
                hidden_size,
            )
        )

        modules.append(
            nn.ReLU()
        )

        current_size = hidden_size

    modules.append(
        nn.Linear(
            current_size,
            3,
        )
    )

    return nn.Sequential(
        *modules
    )


def calculate_siamese_distance(
    encoder,
    x,
):
    color1 = x[:, :3]
    color2 = x[:, 3:]

    embedding1 = encoder(color1)
    embedding2 = encoder(color2)

    return torch.linalg.norm(
        embedding1 - embedding2,
        dim=1,
        keepdim=True,
    )


def calculate_delta_e(x):
    delta_e = torch.linalg.norm(
        x[:, :3] - x[:, 3:],
        dim=1,
        keepdim=True,
    )

    return (
        delta_e - DELTA_E_MIN
    ) / (
        DELTA_E_MAX
        - DELTA_E_MIN
        + 1e-8
    )


def make_loader(
    X,
    Y,
    shuffle,
    noise_std,
    seed,
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        ColorPairDataset(
            X,
            Y,
            noise_std=noise_std,
        ),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        generator=generator,
    )


def cross_entropy_numpy(
    targets,
    predictions,
):
    predictions = np.clip(
        predictions,
        EPSILON,
        1.0,
    )

    return float(
        np.mean(
            -np.sum(
                targets
                * np.log(predictions),
                axis=1,
            )
        )
    )

##5.Method definitions

###5-1.Naive Prediction

In [32]:
def run_naive(
    Y_training,
    Y_testing,
):
    mean_distribution = Y_training.mean(
        axis=0
    )

    predictions = np.repeat(
        mean_distribution.reshape(1, -1),
        len(Y_testing),
        axis=0,
    )

    return (
        cross_entropy_numpy(
            Y_testing,
            predictions,
        ),
        predictions.astype(np.float32),
        mean_distribution.astype(np.float32),
    )

###5-2.ED Regression

In [33]:
def nonlinear_regression_function(
    x,
    coefficient,
    exponent_coefficient,
):
    safe_x = np.maximum(
        x,
        1e-16,
    )

    exponent = np.clip(
        exponent_coefficient / safe_x,
        -8.0,
        8.0,
    )

    return coefficient * np.exp(exponent)


def make_probability_distribution(
    raw_predictions,
):
    predictions = np.asarray(
        raw_predictions,
        dtype=np.float64,
    )

    predictions = np.clip(
        predictions,
        EPSILON,
        None,
    )

    predictions = predictions / np.clip(
        predictions.sum(
            axis=1,
            keepdims=True,
        ),
        EPSILON,
        None,
    )

    return predictions.astype(
        np.float32
    )


def run_regression(
    X_training_raw,
    Y_training,
    X_testing_raw,
    Y_testing,
):
    training_delta_e = np.linalg.norm(
        X_training_raw[:, :3]
        - X_training_raw[:, 3:],
        axis=1,
    )

    testing_delta_e = np.linalg.norm(
        X_testing_raw[:, :3]
        - X_testing_raw[:, 3:],
        axis=1,
    )

    fitted_parameters = []

    for category in range(3):
        try:
            parameters, _ = curve_fit(
                nonlinear_regression_function,
                training_delta_e,
                Y_training[:, category],
                maxfev=(
                    REGRESSION_MAX_FUNCTION_EVALUATIONS
                ),
            )

        except Exception:
            parameters, _ = curve_fit(
                nonlinear_regression_function,
                training_delta_e,
                Y_training[:, category],
                p0=[
                    max(
                        float(
                            Y_training[:, category].mean()
                        ),
                        1e-3,
                    ),
                    -1.0,
                ],
                maxfev=(
                    REGRESSION_MAX_FUNCTION_EVALUATIONS
                    * 2
                ),
            )

        fitted_parameters.append(
            parameters
        )

    testing_raw_predictions = np.stack(
        [
            nonlinear_regression_function(
                testing_delta_e,
                *fitted_parameters[category],
            )
            for category in range(3)
        ],
        axis=1,
    )

    testing_predictions = (
        make_probability_distribution(
            testing_raw_predictions
        )
    )

    return (
        cross_entropy_numpy(
            Y_testing,
            testing_predictions,
        ),
        testing_predictions,
        fitted_parameters,
    )

###5-3.M1:Siamese

In [34]:
class M1Siamese(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = SharedEncoder()
        self.regressor = build_regressor(
            input_size=1
        )

    def forward(self, x):
        distance = calculate_siamese_distance(
            self.encoder,
            x,
        )

        return self.regressor(distance)

###5-4.M2:Siamese+Lab

In [35]:
class M2SiameseLab(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = SharedEncoder()
        self.regressor = build_regressor(
            input_size=7
        )

    def forward(self, x):
        distance = calculate_siamese_distance(
            self.encoder,
            x,
        )

        weighted_lab = (
            x * M2_LAB_WEIGHT
        )

        features = torch.cat(
            [
                distance,
                weighted_lab,
            ],
            dim=1,
        )

        return self.regressor(features)

###5-5.M3:Siamese+ΔE

In [36]:
class M3SiameseDeltaE(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = SharedEncoder()
        self.regressor = build_regressor(
            input_size=2
        )

    def forward(self, x):
        distance = calculate_siamese_distance(
            self.encoder,
            x,
        )

        delta_e = calculate_delta_e(
            x
        )

        features = torch.cat(
            [
                distance,
                delta_e,
            ],
            dim=1,
        )

        return self.regressor(features)

###5-6.M4:Siamese+ΔE+Lab

In [37]:
class M4SiameseDeltaELab(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = SharedEncoder()
        self.regressor = build_regressor(
            input_size=8
        )

    def forward(self, x):
        distance = calculate_siamese_distance(
            self.encoder,
            x,
        )

        delta_e = calculate_delta_e(
            x
        )

        weighted_lab = (
            x * M4_LAB_WEIGHT
        )

        features = torch.cat(
            [
                distance,
                delta_e,
                weighted_lab,
            ],
            dim=1,
        )

        return self.regressor(features)

##6.Training

In [38]:
def evaluate_neural_model(
    model,
    loader,
    return_predictions=False,
):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    probability_batches = []

    with torch.no_grad():
        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            Y_batch = Y_batch.to(DEVICE)

            logits = model(X_batch)
            loss = criterion(logits, Y_batch)

            batch_size = X_batch.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            if return_predictions:
                probability_batches.append(
                    torch.softmax(
                        logits,
                        dim=1,
                    ).cpu().numpy()
                )

    average_loss = total_loss / total_samples

    if return_predictions:
        return (
            average_loss,
            np.concatenate(
                probability_batches,
                axis=0,
            ),
        )

    return average_loss


def train_neural_model(
    model_name,
    model_factory,
):
    #Every model starts from the same fixed seed.
    set_seed(SEED)

    model = model_factory().to(DEVICE)

    training_loader = make_loader(
        X_d1,
        Y_d1,
        shuffle=True,
        noise_std=INPUT_NOISE_STD,
        seed=SEED,
    )

    clean_training_loader = make_loader(
        X_d1,
        Y_d1,
        shuffle=False,
        noise_std=0.0,
        seed=SEED,
    )

    d2_loader = make_loader(
        X_d2,
        Y_d2,
        shuffle=False,
        noise_std=0.0,
        seed=SEED,
    )

    if WEIGHT_DECAY > 0:
        optimizer = optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )
    else:
        optimizer = optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE,
        )

    if USE_SCHEDULER:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=SCHEDULER_FACTOR,
            patience=SCHEDULER_PATIENCE,
            threshold=SCHEDULER_THRESHOLD,
            min_lr=MIN_LEARNING_RATE,
        )
    else:
        scheduler = None

    history_rows = []

    print()
    print("=" * 80)
    print(model_name)

    for epoch in range(1, EPOCHS + 1):
        model.train()

        total_training_loss = 0.0
        total_samples = 0
        epoch_start = time.time()

        for X_batch, Y_batch in training_loader:
            X_batch = X_batch.to(DEVICE)
            Y_batch = Y_batch.to(DEVICE)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, Y_batch)
            loss.backward()
            optimizer.step()

            batch_size = X_batch.size(0)
            total_training_loss += loss.item() * batch_size
            total_samples += batch_size

        noisy_training_loss = (
            total_training_loss
            / total_samples
        )

        if scheduler is not None:
            scheduler.step(noisy_training_loss)

        history_rows.append({
            "Model": model_name,
            "Epoch": epoch,
            "Training loss": noisy_training_loss,
            "Learning rate": optimizer.param_groups[0]["lr"],
        })

        if (
            epoch == 1
            or epoch % PRINT_EVERY == 0
            or epoch == EPOCHS
        ):
            print(
                f"Epoch {epoch:4d} "
                f"train={noisy_training_loss:.6f} "
                f"lr={optimizer.param_groups[0]['lr']:.2e} "
                f"time={time.time() - epoch_start:.3f}s"
            )

    training_loss = evaluate_neural_model(
        model,
        clean_training_loader,
    )

    #D2 is evaluated only once, after the fixed training run finishes.
    (
        d2_loss,
        d2_predictions,
    ) = evaluate_neural_model(
        model,
        d2_loader,
        return_predictions=True,
    )

    if SAVE_MODEL_WEIGHTS:
        safe_model_name = (
            model_name
            .replace("+", "_")
            .replace("Δ", "Delta")
        )

        torch.save(
            model.state_dict(),
            OUTPUT_DIR
            / (
                f"{safe_model_name}_final_state.pt"
            ),
        )

    return (
        training_loss,
        d2_loss,
        d2_predictions,
        pd.DataFrame(history_rows),
        model,
    )


def release_model(model):
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


##7.One-click execution

In [39]:
def run_experiment():
    result_records = []
    history_frames = []
    prediction_frames = []

    #Naive Prediction
    (
        naive_loss,
        naive_predictions,
        naive_distribution,
    ) = run_naive(
        Y_d1,
        Y_d2,
    )

    result_records.append({
        "Method": "Naive Prediction",
        "D2 loss": naive_loss,
    })

    prediction_frames.append(
        pd.DataFrame({
            "Sample index":
                np.arange(len(Y_d2)),
            "Method":
                "Naive Prediction",
            "True 0":
                Y_d2[:, 0],
            "True 1":
                Y_d2[:, 1],
            "True 2":
                Y_d2[:, 2],
            "Predicted 0":
                naive_predictions[:, 0],
            "Predicted 1":
                naive_predictions[:, 1],
            "Predicted 2":
                naive_predictions[:, 2],
        })
    )

    #ED Regression
    (
        regression_loss,
        regression_predictions,
        regression_parameters,
    ) = run_regression(
        X_d1_raw,
        Y_d1,
        X_d2_raw,
        Y_d2,
    )

    result_records.append({
        "Method": "ED Regression",
        "D2 loss": regression_loss,
    })

    prediction_frames.append(
        pd.DataFrame({
            "Sample index":
                np.arange(len(Y_d2)),
            "Method":
                "ED Regression",
            "True 0":
                Y_d2[:, 0],
            "True 1":
                Y_d2[:, 1],
            "True 2":
                Y_d2[:, 2],
            "Predicted 0":
                regression_predictions[:, 0],
            "Predicted 1":
                regression_predictions[:, 1],
            "Predicted 2":
                regression_predictions[:, 2],
        })
    )

    neural_models = [
        (
            "Siamese",
            lambda: M1Siamese(),
        ),
        (
            "Siamese+Lab",
            lambda: M2SiameseLab(),
        ),
        (
            "Siamese+ΔE",
            lambda: M3SiameseDeltaE(),
        ),
        (
            "Siamese+ΔE+Lab",
            lambda: M4SiameseDeltaELab(),
        ),
    ]

    for method_name, model_factory in neural_models:
        (
            training_loss,
            d2_loss,
            d2_predictions,
            history,
            model,
        ) = train_neural_model(
            model_name=method_name,
            model_factory=model_factory,
        )

        result_records.append({
            "Method": method_name,
            "D1 training loss": training_loss,
            "D2 loss": d2_loss,
        })

        history_frames.append(
            history
        )

        prediction_frames.append(
            pd.DataFrame({
                "Sample index":
                    np.arange(len(Y_d2)),
                "Method":
                    method_name,
                "True 0":
                    Y_d2[:, 0],
                "True 1":
                    Y_d2[:, 1],
                "True 2":
                    Y_d2[:, 2],
                "Predicted 0":
                    d2_predictions[:, 0],
                "Predicted 1":
                    d2_predictions[:, 1],
                "Predicted 2":
                    d2_predictions[:, 2],
            })
        )

        release_model(
            model
        )

    result_table = (
        pd.DataFrame(
            result_records
        )
        .set_index(
            "Method"
        )
        .reindex(
            METHOD_ORDER
        )
    )

    result_table.index.name = "Model"

    print()
    print("=" * 80)
    print("Final result table")

    display(
        result_table[
            ["D2 loss"]
        ].round(6)
    )

    histories = pd.concat(
        history_frames,
        ignore_index=True,
    )

    predictions = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    result_table.to_csv(
        OUTPUT_DIR
        / "experiment3_result_table.csv",
    )

    histories.to_csv(
        OUTPUT_DIR
        / "experiment3_training_history.csv",
        index=False,
    )

    if SAVE_PREDICTIONS:
        predictions.to_csv(
            OUTPUT_DIR
            / "experiment3_d2_predictions.csv",
            index=False,
        )

    settings = {
        "D1_PATH":
            D1_PATH,
        "D2_PATH":
            D2_PATH,
        "SEED":
            SEED,
        "NORMALIZATION_REFERENCE":
            NORMALIZATION_REFERENCE,
        "ENCODER_LAYERS":
            ENCODER_LAYERS,
        "REGRESSOR_LAYERS":
            REGRESSOR_LAYERS,
        "EPOCHS":
            EPOCHS,
        "BATCH_SIZE":
            BATCH_SIZE,
        "LEARNING_RATE":
            LEARNING_RATE,
        "INPUT_NOISE_STD":
            INPUT_NOISE_STD,
        "WEIGHT_DECAY":
            WEIGHT_DECAY,
        "M2_LAB_WEIGHT":
            M2_LAB_WEIGHT,
        "M4_LAB_WEIGHT":
            M4_LAB_WEIGHT,
        "USE_SCHEDULER":
            USE_SCHEDULER,
        "naive_distribution":
            naive_distribution.tolist(),
        "regression_parameters": [
            parameters.tolist()
            for parameters
            in regression_parameters
        ],
    }

    with open(
        OUTPUT_DIR
        / "experiment3_settings.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            settings,
            file,
            indent=2,
            ensure_ascii=False,
        )

    return {
        "result_table":
            result_table,
        "histories":
            histories,
        "predictions":
            predictions,
        "settings":
            settings,
    }


experiment3_outputs = (
    run_experiment()
)



Siamese
Epoch    1 train=0.889839 lr=5.00e-04 time=0.436s
Epoch   25 train=0.417080 lr=5.00e-04 time=0.420s
Epoch   50 train=0.395421 lr=5.00e-04 time=0.421s
Epoch   75 train=0.397318 lr=5.00e-04 time=0.411s
Epoch  100 train=0.393655 lr=5.00e-04 time=0.417s
Epoch  125 train=0.397421 lr=5.00e-04 time=0.415s
Epoch  150 train=0.395188 lr=5.00e-04 time=0.415s
Epoch  175 train=0.391262 lr=5.00e-04 time=0.419s
Epoch  200 train=0.389083 lr=5.00e-04 time=0.416s
Epoch  225 train=0.392617 lr=5.00e-04 time=0.411s
Epoch  250 train=0.387768 lr=5.00e-04 time=0.425s
Epoch  275 train=0.391072 lr=5.00e-04 time=0.416s
Epoch  300 train=0.392816 lr=5.00e-04 time=0.422s

Siamese+Lab
Epoch    1 train=0.821942 lr=5.00e-04 time=0.431s
Epoch   25 train=0.404413 lr=5.00e-04 time=0.423s
Epoch   50 train=0.400877 lr=5.00e-04 time=0.433s
Epoch   75 train=0.399970 lr=5.00e-04 time=0.421s
Epoch  100 train=0.396240 lr=5.00e-04 time=0.434s
Epoch  125 train=0.395942 lr=5.00e-04 time=0.432s
Epoch  150 train=0.393501 lr

,D2 loss
Model,
Naive Prediction,0.403033
ED Regression,0.321406
Siamese,0.313173
Siamese+Lab,0.300520
Siamese+ΔE,0.299575
Siamese+ΔE+Lab,0.284998
